# Data preparation

In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

## Prep of tram data

In [28]:
df_tramway = pd.read_csv(r'../Data/tramway_indiv_15min.csv', index_col=0)
df_ref_tram_bus = pd.read_csv(r'../Data/ref_tram_bus.csv',index_col = 0).rename(columns = {'IDT_PNT' : 'VAL_ARRET_CODE','COO_X_WGS84':'lon','COO_Y_WGS84':'lat'})
df_tramway = df_tramway.merge(df_ref_tram_bus, how = 'inner', on = 'VAL_ARRET_CODE')

df_tramway.head()

,LIG_NUMERO_SAE,VAL_ARRET_CODE,CRS_SENS_TRAJET,COD_LIG_CLI,VAL_DATE,Flow,NOM_PNT,NOM_HASTUS,COD_UTE,FLG_PNT_ARR,...,LIB_ADR,COD_INSEE,COD_PST,LIB_CMN,FLG_ADR_EN_FACE,COO_X,COO_Y,lon,lat,CAP
0,520,32102,0,T1,2019-11-02 15:00:00,66,Perrache,NaN,UTT,O,...,24 Crs De Verdun Perrache,69382.0,69002.0,LYON 2EME,N,793765.4,86322.08,4.827012,45.749599,204.0
1,520,32102,0,T1,2019-11-02 17:30:00,53,Perrache,NaN,UTT,O,...,24 Crs De Verdun Perrache,69382.0,69002.0,LYON 2EME,N,793765.4,86322.08,4.827012,45.749599,204.0
2,520,32102,0,T1,2019-11-02 17:45:00,72,Perrache,NaN,UTT,O,...,24 Crs De Verdun Perrache,69382.0,69002.0,LYON 2EME,N,793765.4,86322.08,4.827012,45.749599,204.0
3,520,32102,0,T1,2019-11-02 18:30:00,26,Perrache,NaN,UTT,O,...,24 Crs De Verdun Perrache,69382.0,69002.0,LYON 2EME,N,793765.4,86322.08,4.827012,45.749599,204.0
4,520,32102,0,T1,2019-11-03 04:45:00,1,Perrache,NaN,UTT,O,...,24 Crs De Verdun Perrache,69382.0,69002.0,LYON 2EME,N,793765.4,86322.08,4.827012,45.749599,204.0


### Removing the unwanted stations

In [29]:
NOM_PNT_autriche = ['CDM Saint Priest']
NOM_PNT_norvege = ['Passage machine à laver UTTL', 'Via Part-Dieu Villette V1', 'CDM Meyzieu', 
                   'Retournement Charpennes', 'Retournement SPR Bel Air', 'Retournement Liberté',
                    'Garage Feyssine (voie 16)', 'Garage Feyssine (voie 24)',
                    'Passage machine à laver UTT', 'Retournement Perrache V1',
                    'Retournement Feyssine V2', 'Retournement Quai C. Bernard',
                    'Retournement Perrache V4'
                   ]

df_tramway = df_tramway[~df_tramway.NOM_PNT.isin(NOM_PNT_autriche + NOM_PNT_norvege)]

### Keeping only the T1 and dropping unnecessary columns
We drop rows with C3, RX and TS.
We keep only the T1 rows and the rest will serve to validate the model later.

In [30]:
df_tramway_1 = df_tramway[df_tramway['COD_LIG_CLI']=='T1']
df_tramway_1.columns

Index(['LIG_NUMERO_SAE', 'VAL_ARRET_CODE', 'CRS_SENS_TRAJET', 'COD_LIG_CLI',
       'VAL_DATE', 'Flow', 'NOM_PNT', 'NOM_HASTUS', 'COD_UTE', 'FLG_PNT_ARR',
       'FLG_PNT_DPO', 'FLG_PNT_GAR', 'FLG_PNT_VIA', 'IDT_LIE', 'LIB_ADR',
       'COD_INSEE', 'COD_PST', 'LIB_CMN', 'FLG_ADR_EN_FACE', 'COO_X', 'COO_Y',
       'lon', 'lat', 'CAP'],
      dtype='str')

We drop the unnecessary columns

In [31]:
df_tramway_1 = df_tramway_1.drop(columns= ['NOM_HASTUS',
 'COD_UTE',
 'FLG_PNT_ARR',
 'FLG_PNT_DPO','IDT_LIE',
 'FLG_ADR_EN_FACE', 
 "FLG_PNT_GAR", "FLG_PNT_VIA", 
 "LIB_ADR", "COD_PST", "LIB_CMN", 
 "COD_INSEE", "COO_X", "COO_Y"])
df_tramway_1.columns

Index(['LIG_NUMERO_SAE', 'VAL_ARRET_CODE', 'CRS_SENS_TRAJET', 'COD_LIG_CLI',
       'VAL_DATE', 'Flow', 'NOM_PNT', 'lon', 'lat', 'CAP'],
      dtype='str')

### Removing the COVID period and after and suspicious flow

In [32]:
df_tramway_1['VAL_DATE'] = pd.to_datetime(df_tramway_1['VAL_DATE']) 

In [33]:
# Removing the COVID period
print(
df_tramway_1['VAL_DATE'].describe(),
df_tramway_1['VAL_DATE'].max(),
df_tramway_1['VAL_DATE'].min()
)
# The COVID lockdown started on the 17th march 2020.
df_tramway_1 = df_tramway_1[df_tramway_1['VAL_DATE'] < pd.Timestamp('2020-03-17')]
print(
df_tramway_1['VAL_DATE'].describe(),
df_tramway_1['VAL_DATE'].max(),
df_tramway_1['VAL_DATE'].min()
)

count                        472498
mean     2020-01-11 11:41:10.519028
min             2019-11-01 00:00:00
25%             2019-12-06 00:30:00
50%             2020-01-12 09:15:00
75%             2020-02-15 19:00:00
max             2020-03-30 23:15:00
Name: VAL_DATE, dtype: object 2020-03-30 23:15:00 2019-11-01 00:00:00
count                        450680
mean     2020-01-08 00:01:42.578991
min             2019-11-01 00:00:00
25%             2019-12-04 14:30:00
50%             2020-01-09 00:00:00
75%             2020-02-11 10:45:00
max             2020-03-16 23:45:00
Name: VAL_DATE, dtype: object 2020-03-16 23:45:00 2019-11-01 00:00:00


In [ ]:
# Checking if there are absent hours
df_tramway_1.sort_values('VAL_DATE',inplace=True)

/var/folders/4h/ftzzlfmn3c9bnhtr2nlfsrb40000gn/T/ipykernel_72687/4025616844.py:2: Pandas4Warning: The inplace keyword in DataFrame.sort_values is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_tramway_1.sort_values('VAL_DATE',inplace=True)


NameError: name 'dates' is not defined

In [ ]:

# Removing suspicious flow
df_tramway_1

### Adding columns

#### Columns with no new data
Adding columns for the days of the week, the days of the month, the months, the hours

In [ ]:
df_tramway_1['Day of week'] = pd.to_datetime(df_tramway_1['VAL_DATE']).dt.weekday
df_tramway_1['Day of month'] = pd.to_datetime(df_tramway_1['VAL_DATE']).dt.day
df_tramway_1['Month'] = pd.to_datetime(df_tramway_1['VAL_DATE']).dt.month
df_tramway_1['Hour'] = pd.to_datetime(df_tramway_1['VAL_DATE']).dt.hour

df_tramway_1.head()

In [ ]:
df_tramway_1.to_csv(r'../Cleaned Data/df_tramway_1.csv', index=True)

#### Columns with new data
Creating the columns "holidays", "workdays","peakhours".

In [ ]:
sqnksdj

### Splitting the data

In [ ]:
# Splitting it by direction

In [ ]:
# Splitting it into training and testing

## Now for the weather data

**Links to understand data** :  
DOC POUR LES WEATHER CODES : https://www.historique-meteo.net/assets/docs/weathercodes.txt  
COLONNE OPINION : avis météo de 0 à 4 (0 = très défavorable • 1 = défavorable • 2 = mitigé • 3 = favorable • 4 = très favorable).

### Grouping data and removing unwanted columns

In [ ]:
df_weather_2019 = pd.read_csv(r'../Data/historique-meteo-lyon-2019.csv')
df_weather_2020 = pd.read_csv(r'../Data/historique-meteo-lyon-2020.csv')
df_weather_2020.head()

In [ ]:
df_weather = pd.concat([df_weather_2019, df_weather_2020])
df_weather.columns

In [ ]:
df_weather.drop(columns=['MAX_TEMPERATURE_C', 'MIN_TEMPERATURE_C', 'WINDSPEED_MAX_KMH','PRECIP_TOTAL_DAY_MM','HUMIDITY_MAX_PERCENT', 'VISIBILITY_AVG_KM', 'PRESSURE_MAX_MB',
       'CLOUDCOVER_AVG_PERCENT', 'HEATINDEX_MAX_C', 'DEWPOINT_MAX_C',
       'WINDTEMP_MAX_C','TOTAL_SNOW_MM','UV_INDEX','SUNHOUR',
       'OPINION', 'SUNSET', 'SUNRISE'], inplace=True)
df_weather.head()

### Making it fit the other dataset

In [ ]:
# That part was made using Claude. I provided the structure of my file and asked the code to make it like an example I provided. 
# Map each 6-hour period start to its temperature and weather code columns
period_map = {
    0:  ('TEMPERATURE_MIDNIGHT_0H',  'WEATHER_CODE_MIDNIGHT_0H'),
    6:  ('TEMPERATURE_MORNING_C_6H', 'WEATHER_CODE_MORNING_6H'),
    12: ('TEMPERATURE_NOON_C_12H',   'WEATHER_CODE_NOON_12H'),
    18: ('TEMPERATURE_EVENING_C_18H', 'WEATHER_CODE_EVENING_18H'),
}

daily = df_weather.copy()
daily['DATE'] = pd.to_datetime(daily['DATE'], format='%m-%d-%y')

# Long table: one row per (day, period) with Temperature and weather_code
parts = []
for hour, (t_col, w_col) in period_map.items():
    part = daily[['DATE', t_col, w_col]].copy()
    part.columns = ['DAY', 'Temperature', 'weather_code']
    part['PERIOD'] = hour
    parts.append(part)
values = pd.concat(parts, ignore_index=True)

# Every 15 minutes for every day in the data
times = pd.date_range(
    start=daily['DATE'].min(),
    end=daily['DATE'].max() + pd.Timedelta(hours=23, minutes=45),
    freq='15min',
)
out = pd.DataFrame({'VAL_DATE': times})
out['DAY'] = out['VAL_DATE'].dt.normalize()
out['PERIOD'] = out['VAL_DATE'].dt.hour // 6 * 6

out = out.merge(values, on=['DAY', 'PERIOD'], how='left')
out = out[['VAL_DATE', 'Temperature', 'weather_code']]

# Format dates like the example (11/2/19 15:00)
# On Linux/macOS use %-m and %-d; on Windows use %#m and %#d
out['VAL_DATE'] = out['VAL_DATE'].dt.strftime('%-m/%-d/%y %H:%M')
out.to_csv('../Data/weather_15min_2019_2020.csv', index=False)
print(out.shape)
print(out.head())

In [ ]:
df_weather_daily = out.copy()

### Replacing weather codes
So now, we are going to convert weather codes in ice(3)/snow(2)/rain(1)/no rain no snow(0). using the table weather-codes.csv

In [ ]:
weather_codes = pd.read_csv(r'../Cleaned Data/weather-codes.csv', index_col='WeatherCode')
weather_codes.columns


In [ ]:
df_weather_daily = df_weather_daily.merge(weather_codes, left_on='weather_code', right_index=True, how='left')
df_weather_daily.head()
df_weather_daily.to_csv(r'../Cleaned Data/weather_cleaned.csv',index=False)